# 0.Dependencies

In [1]:
import torch 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
torch.cuda.empty_cache()

In [3]:
!nvidia-smi

Tue Dec  3 12:55:13 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.107.02             Driver Version: 550.107.02     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:65:00.0  On |                  Off |
| 30%   41C    P8             30W /  230W |     966MiB /  24564MiB |     14%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install git+https://github.com/patil-suraj/vqgan-jax.git

  Cloning https://github.com/patil-suraj/vqgan-jax.git to /tmp/pip-req-build-jbmdi3yg
  Running command git clone --filter=blob:none --quiet https://github.com/patil-suraj/vqgan-jax.git /tmp/pip-req-build-jbmdi3yg
  Resolved https://github.com/patil-suraj/vqgan-jax.git to commit 10ef240f8ace869e437f3c32d14898f61512db12
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# 1. Libraries

In [5]:
import io
import os
import shutil
import requests
import time
import numpy as np
from PIL import Image
from math import nan
import math
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, ConcatDataset, DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.cuda.amp import autocast, GradScaler
import jax
import jax.numpy as jnp

import transformers
from transformers.modeling_flax_utils import FlaxPreTrainedModel
from vqgan_jax.modeling_flax_vqgan import VQModel

In [6]:
model_name = "dalle-mini/vqgan_imagenet_f16_16384"
model_vaq = VQModel.from_pretrained(model_name)

# 2. Create Dataloader

In [7]:
# Define custom dataset
class DeepfakeDataset(Dataset):
    def __init__(self, deepfake_dir, source_dir1, source_dir2, transform=None):
        self.deepfake_dir = deepfake_dir
        self.source_dir1 = source_dir1
        self.source_dir2 = source_dir2
        self.transform = transform

        # List all deepfake image filenames
        self.deepfake_images = os.listdir(deepfake_dir)

    def __len__(self):
        return len(self.deepfake_images)

    def __getitem__(self, idx):
        # Get the deepfake image filename
        deepfake_img_name = self.deepfake_images[idx]
        deepfake_img_path = os.path.join(self.deepfake_dir, deepfake_img_name)

        # Extract the source indices from the deepfake filename
        src_idx1, src_idx2 = map(int, deepfake_img_name.split('.')[0].split('_'))

        # Load the corresponding source images
        source_img1_path = os.path.join(self.source_dir1, f"{src_idx1}.png")
        source_img2_path = os.path.join(self.source_dir2, f"{src_idx2}.png")
        deepfake_img = Image.open(deepfake_img_path).convert('RGB')
        source_img1 = Image.open(source_img1_path).convert('RGB')
        source_img2 = Image.open(source_img2_path).convert('RGB')
        
        if self.transform:
            deepfake_img = self.transform(deepfake_img)
            source_img1 = self.transform(source_img1)
            source_img2 = self.transform(source_img2)

        # Return the deepfake image and the tuple of source images
        return deepfake_img, (source_img1, source_img2)

In [8]:
# Define transformations for the images
transform = T.Compose([T.Resize((256, 256)),T.ToTensor()])
batch_size = 32

In [9]:
# Create the dataset and dataloader for training
deepfake_dir_train = 'archive/dataset/train/df'
source_dir1_train = 'archive/dataset/train/src'
source_dir2_train = 'archive/dataset/train/src_remb'
dataset_train = DeepfakeDataset(deepfake_dir_train, source_dir1_train, source_dir2_train, transform=transform)
dataloader_train = DataLoader(dataset_train, batch_size=batch_size,shuffle=True,pin_memory=True,num_workers=4)

In [10]:
# Create the dataset and dataloader for validation
deepfake_dir_valid = 'archive/dataset/valid/df'
source_dir1_valid = 'archive/dataset/valid/src'
source_dir2_valid = 'archive/dataset/valid/src_remb'
dataset_valid = DeepfakeDataset(deepfake_dir_valid,source_dir1_valid, source_dir2_valid,transform=transform)
dataloader_valid = DataLoader(dataset_valid, batch_size=batch_size,shuffle=False)

# 3. Define model_z(i)

In [11]:
class Model_Z(nn.Module):         
    def __init__(self):
        super(Model_Z, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=256, out_channels=2048, kernel_size=3, padding=1)
        self.batchnorm = nn.BatchNorm2d(2048)
        self.conv2 = nn.Conv2d(in_channels=2048, out_channels=256, kernel_size=3, padding=1)
        self.batchnorm2 = nn.BatchNorm2d(256)
        self.conv3 = nn.Conv2d(in_channels=256, out_channels=1024, kernel_size=3, padding=1)
        self.batchnorm3 = nn.BatchNorm2d(1024)
        self.conv4 = nn.Conv2d(in_channels=1024, out_channels=256, kernel_size=3, padding=1)
        self.batchnorm4 = nn.BatchNorm2d(256)
        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.batchnorm5 = nn.BatchNorm2d(512)
        self.conv6 = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=3, padding=1)
        self.batchnorm6 = nn.BatchNorm2d(256)
        self.conv7 = nn.Conv2d(in_channels=256, out_channels=448, kernel_size=3, padding=1)
        self.batchnorm7 = nn.BatchNorm2d(448)
        self.conv8 = nn.Conv2d(in_channels=448, out_channels=384, kernel_size=3, padding=1)
        self.batchnorm8 = nn.BatchNorm2d(384)
        self.conv9 = nn.Conv2d(in_channels=384, out_channels=320, kernel_size=3, padding=1)
        self.batchnorm9 = nn.BatchNorm2d(320)
        self.conv10 = nn.Conv2d(in_channels=320, out_channels=256, kernel_size=3, padding=1)
        self.elu = nn.ELU()

    def forward(self, x):
        res = x
        x = self.elu(self.conv1(x))
        x = self.batchnorm(x)
        x = self.elu(self.conv2(x)) + res
        x = self.batchnorm2(x)
        x = self.elu(self.conv3(x))
        x = self.batchnorm3(x)
        x = self.elu(self.conv4(x)) + res
        x = self.batchnorm4(x)
        x = self.elu(self.conv5(x))
        x = self.batchnorm5(x)
        x = self.elu(self.conv6(x)) + res
        x = self.batchnorm6(x)
        x = self.elu(self.conv7(x))
        x = self.batchnorm7(x)
        x = self.elu(self.conv8(x))
        x = self.batchnorm8(x)
        x = self.elu(self.conv9(x))
        x = self.batchnorm9(x)
        out = self.elu(self.conv10(x)) + res
        return out


In [12]:
class Model_Z1(nn.Module):         
    def __init__(self):
        super(Model_Z1, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=256, out_channels=2048, kernel_size=3, padding=1)
        self.batchnorm = nn.BatchNorm2d(2048)
        self.conv2 = nn.Conv2d(in_channels=2048, out_channels=256, kernel_size=3, padding=1)
        self.batchnorm2 = nn.BatchNorm2d(256)
        self.conv3 = nn.Conv2d(in_channels=256, out_channels=1024, kernel_size=3, padding=1)
        self.batchnorm3 = nn.BatchNorm2d(1024)
        self.conv4 = nn.Conv2d(in_channels=1024, out_channels=256, kernel_size=3, padding=1)
        self.batchnorm4 = nn.BatchNorm2d(256)
        self.conv5 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, padding=1)
        self.batchnorm5 = nn.BatchNorm2d(512)
        self.conv6 = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=3, padding=1)
        
        self.elu = nn.ELU()

    def forward(self, x):
        res = x
        x = self.elu(self.conv1(x))
        x = self.batchnorm(x)
        x = self.elu(self.conv2(x)) + res
        x = self.batchnorm2(x)
        x = self.elu(self.conv3(x))
        x = self.batchnorm3(x)
        x = self.elu(self.conv4(x)) + res
        x = self.batchnorm4(x)
        x = self.elu(self.conv5(x))
        x = self.batchnorm5(x)
        out = self.elu(self.conv6(x)) + res
        return out


In [13]:
model_z1 = Model_Z1()
model_z1 = model_z1.to(device)
model_z1.load_state_dict(torch.load("model/model_z1.pth",map_location=device),strict=False)

model_z2 = Model_Z()
model_z2 = model_z2.to(device)
model_z2.load_state_dict(torch.load("model/model_z2.pth",map_location=device),strict=False)

model_zdf = Model_Z()
model_zdf = model_zdf.to(device)
model_zdf.load_state_dict(torch.load("model/model_zdf.pth",map_location=device),strict=False)


<All keys matched successfully>

_IncompatibleKeys(missing_keys=['batchnorm4.weight', 'batchnorm4.bias', 'batchnorm4.running_mean', 'batchnorm4.running_var', 'conv5.weight', 'conv5.bias', 'batchnorm5.weight', 'batchnorm5.bias', 'batchnorm5.running_mean', 'batchnorm5.running_var', 'conv6.weight', 'conv6.bias'], unexpected_keys=[])


In [14]:
"""for t in model_z1.parameters():
    print(t.shape)"""

'for t in model_z1.parameters():\n    print(t.shape)'

# 3.1.Pruning

In [15]:
import torch.nn.utils.prune as prune

In [16]:
# Apply pruning to all convolutional layers
for name, module in model_z1.named_modules():
    if isinstance(module, nn.Conv2d):
        prune.l1_unstructured(module, name="weight", amount=0.1)  # Prune 10% of weights

"""# Verify pruning masks
print("Pruning applied:")
for name, module in model_z1.named_modules():
    if isinstance(module, nn.Conv2d):
        print(f"{name}: {list(module.named_buffers())}")"""

# Remove pruning reparameterization (optional)
for name, module in model_z1.named_modules():
    if isinstance(module, nn.Conv2d):
        prune.remove(module, "weight")


In [17]:
# Apply pruning to all convolutional layers
for name, module in model_z2.named_modules():
    if isinstance(module, nn.Conv2d):
        prune.l1_unstructured(module, name="weight", amount=0.1)  # Prune 10% of weights

# Remove pruning reparameterization (optional)
for name, module in model_z2.named_modules():
    if isinstance(module, nn.Conv2d):
        prune.remove(module, "weight")


# 4. Tensor and Jax array conversion

In [18]:
def tensor_jax(x):
    x_np = x.detach().permute(0, 2, 3, 1).cpu().numpy()  # Convert from (N, C, H, W) to (N, H, W, C) and move to CPU
    x_jax = jnp.array(x_np)
    return x_jax

In [19]:
def jax_to_tensor(x):
    x_tensor = torch.tensor(np.array(x),requires_grad=True).permute(0, 3, 1, 2).to(device)  # Convert from (N, H, W, C) to (N, C, H, W)
    return x_tensor

# 5. Define validation and training

In [20]:
# Function to save data to pickle file
def train_history(hist_train, filename='train_history_134.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(hist_train, f)

hist_train = {"epoch":[], "loss_z1":[], "loss_z2":[], "loss_df_z":[], "loss_img1":[], "loss_img2":[], "loss_df_img":[]}

In [21]:
# Function to save data to pickle file
def valid_history(hist_valid, filename='valid_history_134.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(hist_valid, f)

hist_valid = {"epoch":[], "loss_z1":[], "loss_z2":[], "loss_block":[],"loss_img1":[], "loss_img2":[], "loss_rec":[],"total_loss":[]}

In [22]:
def valid_z(model_vaq, model_z1, model_z2, model_zdf, dataloader):
    criterion = nn.MSELoss()
    model_z1.eval()
    model_z2.eval()
    model_zdf.eval()

    with torch.no_grad():
        running_loss_z1 = 0.0
        running_loss_z2 = 0.0
        running_loss_zdf = 0.0
        running_loss_img1 = 0.0
        running_loss_img2 = 0.0
        running_loss_dfimg = 0.0
        
        for df_img,(img1,img2) in dataloader:
            df_img = df_img.to(device)
            img1 = img1.to(device)
            img2 = img2.to(device)
            
            #convert images: tensor --> jax_array
            df_img_jax = tensor_jax(df_img)
            img1_jax = tensor_jax(img1)
            img2_jax = tensor_jax(img2)
            
            #calculate quantized_block for all images
            z_df,_ = model_vaq.encode(df_img_jax) 
            z_img1,_ = model_vaq.encode(img1_jax)
            z_img2,_ = model_vaq.encode(img2_jax)
            
            #convert quantized_block: jax_array --> tensor
            z_df_tensor = jax_to_tensor(z_df)
            z_img1_tensor = jax_to_tensor(z_img1)
            z_img2_tensor = jax_to_tensor(z_img2)
            
            ##----------------------------------------------------------------------
            ##----------------------model_Z1-----------------------
            outputs_z1 = model_z1(z_df_tensor) # F(Z)-> Z11
            loss1 = criterion(outputs_z1, z_img1_tensor)
            z1_rec_jax = tensor_jax(outputs_z1)
            rec_img1 = model_vaq.decode(z1_rec_jax)
            rec_img1_tensor = jax_to_tensor(rec_img1)
            imgloss1 = criterion(rec_img1_tensor, img1)

            running_loss_z1 += loss1.item()
            running_loss_img1 += imgloss1.item()
            ##----------------------------------------------------------------------
            ##----------------------model_Z2-----------------------
            outputs_z2 = model_z2(z_df_tensor) # G(Z)-> Z22
            loss2 = criterion(outputs_z2, z_img2_tensor)
            z2_rec_jax = tensor_jax(outputs_z2)
            rec_img2 = model_vaq.decode(z2_rec_jax)
            rec_img2_tensor = jax_to_tensor(rec_img2)
            imgloss2 = criterion(rec_img2_tensor, img2)

            running_loss_z2 += loss2.item()
            running_loss_img2 += imgloss2.item()
            ##----------------------------------------------------------------------
            ##----------------------model_Z_df-----------------------
            z_rec = outputs_z1 + outputs_z2 # Create quantized block, Z_rec = Z11 + Z22
            z_rec = z_rec.to(device)
            outputs_zdf = model_zdf(z_rec) # H(Z_rec) -> Z_df
            lossdf = criterion(outputs_zdf, z_df_tensor) # loss(Z_df,Z)
            running_loss_zdf += lossdf.item()
            #calculate generated_DeepFake_image reconstruction loss
            zdf_rec_jax = tensor_jax(outputs_zdf)
            rec_df = model_vaq.decode(zdf_rec_jax) #VQGAN_Decoder, D(Z_df) -> Generated DeepFake image from quantized block Z_df
            rec_df_tensor = jax_to_tensor(rec_df)
            dfimgloss = criterion(rec_df_tensor, df_img) #calculate reconstruction_loss of images between generated_DeepFake_image and DeepFake_image
            running_loss_dfimg += dfimgloss.item()


        ##---------------------Validation summary------------------------------------
        loss_z1 = running_loss_z1 / len(dataloader)
        loss_img1 = running_loss_img1 / len(dataloader)
        loss_z2 = running_loss_z2 / len(dataloader)
        loss_img2 = running_loss_img2 / len(dataloader)
        loss_zdf = running_loss_zdf / len(dataloader)
        loss_dfimg = running_loss_dfimg / len(dataloader)
        hist_valid["loss_z1"].append(loss_z1)
        hist_valid["loss_z2"].append(loss_z2)
        hist_valid["loss_block"].append(loss_zdf)
        hist_valid["loss_img1"].append(loss_img1)
        hist_valid["loss_img2"].append(loss_img2)
        hist_valid["loss_rec"].append(loss_dfimg)
        hist_valid["total_loss"].append(loss_zdf + loss_dfimg)
        valid_history(hist_valid)
        
        print("Validation:")
        print(f"Loss-z1: {loss_z1:.4f}, Loss-z2: {loss_z2:.4f}, Loss-dfz:{loss_zdf:.4f}, Loss-img1_rec: {loss_img1:.5f}, Loss-img2_rec: {loss_img2:.5f}, Loss-df_rec:{loss_dfimg:.5f}, Total-loss:{loss_zdf+loss_dfimg:4f}")
        
    return "Done!!"

In [23]:
def train_z(model_vaq, model_z1, model_z2, model_zdf, dataloader, dataloader_valid, lrs, num_epochs):
    criterion = nn.MSELoss()
    model_z1.train()
    optimizer1 = optim.Adam(model_z1.parameters(), lr=5e-5)
    model_z2.train()
    optimizer2 = optim.Adam(model_z2.parameters(), lr=lrs)
    model_zdf.train()
    optimizer_df = optim.Adam(model_zdf.parameters(), lr=lrs)

    # Initialize the GradScaler for automatic mixed precision
    scaler = GradScaler()

    for epoch in range(num_epochs):
        start_time = time.time()
        running_loss_z1 = 0.0
        running_loss_z2 = 0.0
        running_loss_zdf = 0.0
        running_loss_img1 = 0.0
        running_loss_img2 = 0.0
        running_loss_dfimg = 0.0
        for df_img, (img1,img2) in dataloader:
            df_img = df_img.to(device)
            img1 = img1.to(device)
            img2 = img2.to(device)

            # convert images: tensor --> jax_array
            df_img_jax = tensor_jax(df_img)
            img1_jax = tensor_jax(img1)
            img2_jax = tensor_jax(img2)
            
            # calculate quantized_code(z) for all images
            z_df,_ = model_vaq.encode(df_img_jax)
            z_img1,_ = model_vaq.encode(img1_jax)
            z_img2,_ = model_vaq.encode(img2_jax)
            
            # convert quantized_code(z): jax_array --> tensor
            z_df_tensor = jax_to_tensor(z_df)
            z_img1_tensor = jax_to_tensor(z_img1)
            z_img2_tensor = jax_to_tensor(z_img2)

            ##----------------------model_z1-----------------------
            optimizer1.zero_grad()

            with autocast():  # Enable mixed precision
                outputs_z1 = model_z1(z_df_tensor)
                loss1 = criterion(outputs_z1, z_img1_tensor)
                z1_rec_jax = tensor_jax(outputs_z1)
                rec_img1 = model_vaq.decode(z1_rec_jax)
                rec_img1_tensor = jax_to_tensor(rec_img1)
                imgloss1 = criterion(rec_img1_tensor, img1)

            # Scale the loss before backward pass
            scaler.scale(loss1).backward()
            scaler.step(optimizer1)
            scaler.update()

            running_loss_z1 += loss1.item()
            running_loss_img1 += imgloss1.item()

            ##----------------------model_z2-----------------------
            optimizer2.zero_grad()

            with autocast():  # Enable mixed precision
                outputs_z2 = model_z2(z_df_tensor)
                loss2 = criterion(outputs_z2, z_img2_tensor)
                z2_rec_jax = tensor_jax(outputs_z2)
                rec_img2 = model_vaq.decode(z2_rec_jax)
                rec_img2_tensor = jax_to_tensor(rec_img2)
                imgloss2 = criterion(rec_img2_tensor, img2)

            # Scale the loss before backward pass
            scaler.scale(loss2).backward()
            scaler.step(optimizer2)
            scaler.update()

            running_loss_z2 += loss2.item()
            running_loss_img2 += imgloss2.item()

            ##----------------------model_zdf-----------------------
            z_rec = z_img1_tensor + z_img2_tensor
            z_rec = z_rec.to(device)
            optimizer_df.zero_grad()

            with autocast():  # Enable mixed precision
                outputs_zdf = model_zdf(z_rec)
                lossdf = criterion(outputs_zdf, z_df_tensor)
                zdf_rec_jax = tensor_jax(outputs_zdf)
                rec_df = model_vaq.decode(zdf_rec_jax)
                rec_df_tensor = jax_to_tensor(rec_df)
                dfimgloss = criterion(rec_df_tensor, df_img)

            # Scale the loss before backward pass
            scaler.scale(lossdf).backward()
            scaler.step(optimizer_df)
            scaler.update()

            running_loss_zdf += lossdf.item()
            running_loss_dfimg += dfimgloss.item()
            

        ##---------------------Epoch summary------------------------------------
        loss_z1 = running_loss_z1 / len(dataloader)
        loss_z2 = running_loss_z2 / len(dataloader)
        loss_dfz = running_loss_zdf / len(dataloader)
        loss_img1 = running_loss_img1 / len(dataloader)
        loss_img2 = running_loss_img2 / len(dataloader)
        loss_dfimg = running_loss_dfimg / len(dataloader)

        path_z1 = "model_z1_"+str(epoch)+".pth"
        path_z2 = "model_z2_"+str(epoch)+".pth"
        path_zdf = "model_zdf_"+str(epoch)+".pth"
        torch.save(model_z1.state_dict(), path_z1)
        torch.save(model_z2.state_dict(), path_z2)
        torch.save(model_zdf.state_dict(), path_zdf)

        
        hist_train["epoch"].append(epoch+1)
        hist_train["loss_z1"].append(loss_z1)
        hist_train["loss_z2"].append(loss_z2)
        hist_train["loss_df_z"].append(loss_dfz)
        hist_train["loss_img1"].append(loss_img1)
        hist_train["loss_img2"].append(loss_img2)
        hist_train["loss_df_img"].append(loss_dfimg)
        train_history(hist_train)
        
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print("Training:")
        print(f"Loss-z1: {loss_z1:.4f}, Loss-z2: {loss_z2:.4f}, Loss-dfz:{loss_dfz:.4f}, Loss-img1_rec: {loss_img1:.5f}, Loss-img2_rec: {loss_img2:.5f}, Loss-df_rec:{loss_dfimg:.5f}")
        msg = valid_z(model_vaq, model_z1, model_z2, model_zdf, dataloader_valid)
        epoch_time = (time.time() - start_time) / 3600
        print(f"Time:{epoch_time:.2f} hrs")

    return msg


# 6. Training

In [24]:
learning_rate = 3e-4
num_epochs = 7

In [ ]:
msg = train_z(model_vaq, model_z1, model_z2, model_zdf, dataloader_train, dataloader_valid, learning_rate, num_epochs)

Epoch [1/7]
Training:
Loss-z1: 0.0510, Loss-z2: 0.3061, Loss-dfz:0.1294, Loss-img1_rec: 0.00628, Loss-img2_rec: 0.02980, Loss-df_rec:0.01113
Validation:
Loss-z1: 0.0717, Loss-z2: 0.5128, Loss-dfz:0.3056, Loss-img1_rec: 0.00671, Loss-img2_rec: 0.06186, Loss-df_rec:0.02811, Total-loss:0.333744
Time:3.69 hrs
Epoch [2/7]
Training:
Loss-z1: 0.0501, Loss-z2: 0.2993, Loss-dfz:0.1263, Loss-img1_rec: 0.00627, Loss-img2_rec: 0.02804, Loss-df_rec:0.01075
Validation:
Loss-z1: 0.0708, Loss-z2: 0.5138, Loss-dfz:0.3166, Loss-img1_rec: 0.00672, Loss-img2_rec: 0.06074, Loss-df_rec:0.03629, Total-loss:0.352900
Time:3.68 hrs
Epoch [3/7]
Training:
Loss-z1: 0.0497, Loss-z2: 0.2961, Loss-dfz:0.1252, Loss-img1_rec: 0.00626, Loss-img2_rec: 0.02754, Loss-df_rec:0.01068
Validation:
Loss-z1: 0.0721, Loss-z2: 0.5172, Loss-dfz:0.3054, Loss-img1_rec: 0.00676, Loss-img2_rec: 0.06214, Loss-df_rec:0.03391, Total-loss:0.339316
Time:3.69 hrs
Epoch [4/7]
Training:
Loss-z1: 0.0494, Loss-z2: 0.2939, Loss-dfz:0.1244, Loss-i

# 7.Result